In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')

In [ ]:
data_path = "../data/processed/train_pairs.csv"
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)
df['prompt'] = df['prompt'].fillna("")
df['response'] = df['response'].fillna("")
print(f"Dataset shape: {df.shape}")
display(df.head())

Loading data from ../data/processed/train_pairs.csv...
Dataset shape: (387431, 2)


,prompt,response
0,"Say , Jim , how about going for a few beers af...",You know that is tempting but is really not go...
1,"Say , Jim , how about going for a few beers af...",You know that is tempting but is really not go...
2,You know that is tempting but is really not go...,What do you mean ? It will help us to relax .
3,"Say , Jim , how about going for a few beers af...",You know that is tempting but is really not go...
4,You know that is tempting but is really not go...,What do you mean ? It will help us to relax .


In [ ]:
print("Initializing TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(stop_words='english')
print("Fitting vectorizer and transforming queries into vectors... (This may take a moment)")
tfidf_matrix = vectorizer.fit_transform(df['prompt'])
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

Initializing TF-IDF Vectorizer...
Fitting vectorizer and transforming queries into vectors... (This may take a moment)
Vocabulary size: 16577
TF-IDF Matrix shape: (387431, 16577)


In [ ]:
def retrieve_response(user_query, top_k=1):
    query_vec = vectorizer.transform([user_query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    best_match_indices = similarities.argsort()[-top_k:][::-1]
    results = []
    for idx in best_match_indices:
        score = similarities[idx]
        if score > 0:
            match_query = df.iloc[idx]['prompt']
            response = df.iloc[idx]['response']
            results.append({
                'matched_query': match_query,
                'response': response,
                'score': score
            })
            
    return results
sample_q = "how about going for a few beers?"
print(f"Test Query: '{sample_q}'\n")
res = retrieve_response(sample_q)
if res:
    print(f"Matched Database Query: {res[0]['matched_query']}")
    print(f"System Response: {res[0]['response']}")
    print(f"Similarity Score: {res[0]['score']:.4f}")
else:
    print("No relevant match found.")

Test Query: 'how about going for a few beers?'

Matched Database Query: I usually only have 2 beers .
System Response: You're a light weight .
Similarity Score: 0.7551


In [ ]:
test_queries = [
    "Hello, how are you doing today?",
    "Can you recommend a good restaurant?",
    "I am feeling a bit sad right now.",
    "What time is the flight?",
    "Exsanguination protocols in medical emergencies" # A complex query it likely won't know
]
print("--- TF-IDF Baseline Demonstration ---\n")
for q in test_queries:
    print(f"User Query: {q}")
    results = retrieve_response(q, top_k=1)
    if results:
        best_match = results[0]
        print(f"Bot Response: {best_match['response']}")
        print(f"(Matched against: '{best_match['matched_query']}' | Score: {best_match['score']:.4f})")
    else:
        print("Bot Response: I'm sorry, I don't understand.")
    print("-" * 50)

--- TF-IDF Baseline Demonstration ---

User Query: Hello, how are you doing today?
Bot Response: Very well . Thank you .
(Matched against: 'How are you doing today ?' | Score: 0.8261)
--------------------------------------------------
User Query: Can you recommend a good restaurant?
Bot Response: This one from Sony gives a very sharp picture .
(Matched against: 'No . Which one do you recommend ?' | Score: 0.6687)
--------------------------------------------------
User Query: I am feeling a bit sad right now.
Bot Response: Her parents scolded her severely and she's very depressed now .
(Matched against: 'She must be very sad .' | Score: 0.6436)
--------------------------------------------------
User Query: What time is the flight?
Bot Response: It's at 7:30 a . m .
(Matched against: 'What time's your flight ?' | Score: 1.0000)
--------------------------------------------------
User Query: Exsanguination protocols in medical emergencies
Bot Response: I will try . My apologies .
(Matched 

In [ ]:
import os
import pickle
model_dir = "../models/tfidf"
os.makedirs(model_dir, exist_ok=True)
print(f"Saving models to {model_dir}...")
with open(os.path.join(model_dir, "vectorizer.pkl"), "wb") as f:
    pickle.dump(vectorizer, f)
with open(os.path.join(model_dir, "tfidf_matrix.pkl"), "wb") as f:
    pickle.dump(tfidf_matrix, f)
print("Baseline model successfully saved!")

Saving models to ../models/tfidf...
Baseline model successfully saved!
